# JAX: transformations of functions

Section 15.3 of the notes.

`grad`, `vmap` and `jit` take a function and return a function. The price is
that the function must be traceable: no state, no branching on values, and
arrays that are never modified in place.

In [ ]:
import time
import timeit

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)


def f(x):
    return jnp.sin(x**2)


print("f'(1)  ", jax.grad(f)(1.0))
print("f''(1) ", jax.grad(jax.grad(f))(1.0))
print("batch  ", jax.vmap(jax.grad(f))(jnp.linspace(0, 2, 5)))

## 1. Arrays are not modified

An update returns a new array. The old one is unchanged, which is what allows
the same trace to be reused.

In [ ]:
x = jnp.arange(5.0)
y = x.at[0].set(99.0)
print("x", x)
print("y", y)

## 2. Randomness is explicit

There is no hidden generator. A key is passed in and split, so a function that
draws random numbers stays a function of its arguments.

In [ ]:
key = jax.random.key(0)
k1, k2 = jax.random.split(key)
print(
    "same key, same numbers:",
    jnp.allclose(jax.random.normal(k1, (3,)), jax.random.normal(k1, (3,))),
)
print(
    "different keys:        ", jax.random.normal(k1, (3,)), jax.random.normal(k2, (3,))
)

## 3. What `jit` costs and saves

The first call traces and compiles. Later calls run compiled code. A change of
shape or type triggers a new compilation.

In [ ]:
def step(w, x):
    for _ in range(20):
        w = w - 0.01 * jnp.tanh(w @ x) @ x.T
    return w


rng = jax.random.key(1)
w = jax.random.normal(rng, (64, 64))
x = jax.random.normal(jax.random.split(rng)[0], (64, 64))

step_jit = jax.jit(step)
t0 = time.perf_counter()
step_jit(w, x).block_until_ready()
print(f"first call (traces and compiles): {1e3 * (time.perf_counter() - t0):.0f} ms")

t_plain = (
    min(timeit.repeat(lambda: step(w, x).block_until_ready(), repeat=3, number=5)) / 5
)
t_jit = (
    min(timeit.repeat(lambda: step_jit(w, x).block_until_ready(), repeat=3, number=5))
    / 5
)
print(f"without jit {1e6 * t_plain:8.1f} us")
print(f"with jit    {1e6 * t_jit:8.1f} us   ratio {t_plain / t_jit:.1f}x")

## 4. Branching on a value

A Python `if` on the value of an argument cannot be traced, because the trace
must be the same for every value. The error says so, and `jax.lax.cond` is the
way to write it.

In [ ]:
def bad(x):
    if x > 0:  # depends on the value, not on the shape
        return x**2
    return -x


try:
    jax.jit(bad)(1.0)
except jax.errors.TracerBoolConversionError as err:
    print("failed:", str(err).splitlines()[0])


def good(x):
    return jax.lax.cond(x > 0, lambda v: v**2, lambda v: -v, x)


print("lax.cond:", jax.jit(good)(1.0), jax.jit(good)(-2.0))

## 5. The stencil of Chapter 2, compiled

`jit` applies to any traceable function, not only to networks. Here it compiles
one step of the method of lines.

In [ ]:
D_COEFF = 2e-5


@jax.jit
def euler_step(u, dt, dx):
    lap = (jnp.roll(u, -1) - 2 * u + jnp.roll(u, 1)) / dx**2
    return u + dt * (D_COEFF * lap + u - u**3)


d = 2048
xs = -1 + 2 * jnp.arange(d) / d
u0 = xs**2 * jnp.cos(jnp.pi * xs)
u = euler_step(u0, 0.005, 2 / d).block_until_ready()
print("shape", u.shape, "| max |u|", float(jnp.abs(u).max()))